# 02 · 位置编码与网络层

“Tom 追 Jerry”和“Jerry 追 Tom”，用的是同样三个词，故事却完全反过来了。上一篇的注意力让词之间交换信息；这一篇继续回答：**模型怎样知道顺序，又怎样把交换来的信息加工成预测？**

我们给词加上位置，再接上 FFN、残差和归一化，最后搭出 Encoder、Decoder 和词表出口。每块积木都用小张量观察，方便跟着数据走一遍。

在 `transformer` 环境里从第一格运行即可。完整路线见[第一篇](01_注意力与多头机制.ipynb)，启动方法见[仓库入口](../README.md)。

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)
import copy

本篇把向量宽度设得很小，让你能直接看清每一列。演示中的 `eval()` 会关闭 Dropout，使前向计算更方便比较。

## 1. 同一个词，怎样知道自己坐在哪？

Embedding 按 token 编号查表。同一个词连续出现四次，会查出四条相同的向量。我们给每个位置再加一张“座位标签”：

$$\text{输入表示}=\text{词向量}+\text{位置编码}.$$

标签用一组快慢不同的时钟来写。某些维度像秒针，邻近位置就变化明显；另一些像时针，在更长的范围内慢慢变化：

$$PE(p,2i)=\sin(p/10000^{2i/D}),\quad PE(p,2i+1)=\cos(p/10000^{2i/D}).$$

这里 p 是位置，D 是向量宽度，i 标记第几对 sin/cos。把这些不同频率的读数组合起来，每个位置就有了自己的模式。

<details>
<summary>为什么纯注意力需要这张标签？</summary>

在没有位置输入和位置相关掩码时，打乱输入会让自注意力的输出按同样顺序打乱，这叫置换等变性。加入位置编码后，词向量的内容和它所处的位置一起参与匹配。

固定偏移 k 时，每对 sin/cos 可通过一个旋转矩阵把 p 的编码变到 p+k。这给学习相对位置提供了便利。本实现预先保存 `max_len` 个位置，示例使用偶数维 D。

</details>

### 实现：词语查表，再加上位置读数

`Embeddings` 取出词向量，`PositionalEncoding` 取出对应的座位标签，两者相加。
位置表用 `register_buffer` 保存：跟着模型一起保存、移动设备，数值由公式生成。

定义完类后，下面会给四个相同 token 加标签，并画出位置编码热力图。

In [ ]:
class Embeddings(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.lut = nn.Embedding(vocab, d_model)
        self.d_model = d_model
    def forward(self, x):
        return self.lut(x) * math.sqrt(self.d_model)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout, max_len=5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)
        self.pe: torch.Tensor
    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [ ]:
d_model = 8
tokens = torch.tensor([[3, 3, 3, 3]])
embedding = Embeddings(d_model, vocab=16).eval()
position = PositionalEncoding(d_model, dropout=0.0, max_len=32).eval()
with torch.no_grad():
    embedded = embedding(tokens)
    positioned = position(embedded)
print("Embedding shape:", embedded.shape)
print("Same token before position:", embedded[0, :2])
print("Same token after position:", positioned[0, :2])
plt.figure(figsize=(9, 3))
plt.imshow(position.pe[0, :16].numpy(), aspect="auto", cmap="coolwarm")
plt.xlabel("Feature dimension")
plt.ylabel("Position")
plt.title("Sinusoidal positional encoding")
plt.colorbar()
plt.show()

看打印出的前两行：加标签前，它们一样；加标签后，它们有了区别。热力图中横向不同的列对应不同频率，纵向则是位置逐步往后移动。

## 2. 交流之后，还要加工：FFN 与 Add & Norm

注意力像开会，让每个位置收集相关信息。FFN 像回到各自工位，用同一套规则消化收集来的内容：

$$FFN(x)=\operatorname{ReLU}(xW_1+b_1)W_2+b_2.$$

两层线性变换通常走 `D → 4D → D`：先展开成更多中间特征，经过 ReLU，再压回原来的宽度。每个位置分别处理，参数在位置之间共享。

### 残差：在原稿上写修改量

如果子层算出 F(x)，残差把它加回原稿：$y=x+F(x)$。子层可以专心学习“需要补充什么”。这条直连也给梯度提供了更短的传播路径。

### LayerNorm：把一行特征调到合适的尺度

对一个 token 的 D 个特征，先减去均值，再除以标准差，最后由可学习的 γ、β 调整幅度和偏移：

$$LN(x)=\gamma\frac{x-\mu}{\sqrt{\operatorname{Var}(x)+\epsilon}}+\beta.$$

下面把这几步串成 `LN(x + Dropout(Sublayer(x)))`。先相加、后归一化，叫 **Post-LN**；另一个常见顺序 **Pre-LN** 是 `x + Dropout(Sublayer(LN(x)))`。

### 实现：沿着三处名字读代码

`w_1` 和 `w_2` 对应 FFN 的展开与压缩；`a_2`、`b_2` 对应归一化后的缩放与平移；`SublayerConnection` 负责把子层结果加回 x。

方差沿最后一维计算，`unbiased=False` 表示对这一整行特征求总体方差。eps 放在平方根内，让分母在方差很小时仍保持稳定。

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.w_1 = nn.Linear(d_model, d_ff)
        self.w_2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.w_2(self.dropout(F.relu(self.w_1(x))))


class LayerNorm(nn.Module):
    def __init__(self, features, eps=1e-6):
        super().__init__()
        self.a_2 = nn.Parameter(torch.ones(features))
        self.b_2 = nn.Parameter(torch.zeros(features))
        self.eps = eps
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return self.a_2 * (x - mean) / torch.sqrt(variance + self.eps) + self.b_2


class SublayerConnection(nn.Module):
    def __init__(self, size, dropout):
        super().__init__()
        self.norm = LayerNorm(size)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, sublayer):
        return self.norm(x + self.dropout(sublayer(x)))

In [ ]:
ffn = PositionWiseFeedForward(d_model, d_ff=32, dropout=0.0).eval()
connection = SublayerConnection(d_model, dropout=0.0).eval()
norm = LayerNorm(d_model)
with torch.no_grad():
    processed = ffn(positioned)
    normalized = norm(positioned)
    connected = connection(positioned, ffn)
print("FFN input/output:", positioned.shape, processed.shape)
print("Normalized mean:", normalized.mean(dim=-1))
print("Normalized variance:", normalized.var(dim=-1, unbiased=False))
print("Residual connection output:", connected.shape)

运行后看两件事：FFN 的输入输出形状一致；LayerNorm 后每行均值接近 0、方差接近 1。eps 会让方差略偏离 1，训练还会继续调整 γ 和 β。

## 3. 给 Decoder 一张“只能看前面”的座位表

要写出“我 爱 猫”，模型先看到 BOS 来预测“我”，看到“BOS 我”来预测“爱”，再看到“BOS 我 爱”来预测“猫”。

训练时可以把右移后的输入一起放进去，再用 **因果掩码** 规定每个位置能看到哪里：

```text
输入位置   BOS  我  爱
BOS         ✓   ·   ·
我          ✓   ✓   ·
爱          ✓   ✓   ✓
```

下三角表示可见位置。本实现用 `True` 表示可见，`masked_fill(mask == 0, ...)` 把其余分数压低，使它们在 Softmax 后的权重为 0。

In [ ]:
length = 5
causal_mask = torch.ones(length, length, dtype=torch.bool).tril()
print(causal_mask)
plt.figure(figsize=(4, 4))
plt.imshow(causal_mask.numpy(), vmin=0, vmax=1, cmap="Greys")
plt.xlabel("Key position")
plt.ylabel("Query position")
plt.title("Allowed attention positions")
plt.show()

图里的可见区域像一段楼梯：越往后走，已经拿到的前缀越长。这样，训练时每个位置读到的内容就和逐词生成时对应起来了。

## 4. 三个角色接力：读原文、写前缀、选下一个词

把翻译过程想成一位译者：

1. **Encoder 读原文**，把源序列变成一排带上下文的向量，称为 memory。
2. **Decoder 看已写前缀**，通过自注意力整理已有内容，再带着 Q 查询 memory 中的 K/V。
3. **Generator 给词表打分**，把 Decoder 的隐藏向量变成每个候选 token 的对数概率。

Encoder 的一层包含自注意力和 FFN，Decoder 再加入交叉注意力。每个子层都由 Add & Norm 连接。`clones` 用深拷贝为各层建立独立参数。

接下来用随机初始化的小模型追踪形状：源句子进入 memory，目标前缀得到隐藏状态，最后来到词表出口。

### 实现：先把 Encoder 接好

从 `MultiHeadAttention.forward` 往下读，能看到上一篇的分头、匹配和汇总。`EncoderLayer` 再把它与 FFN 连成一层，`Encoder` 把这样的层堆起来。

自注意力调用中的 `x, x, x` 分别提供 Q、K、V。

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, q, k, v, mask=None):
        batch_size = q.size(0)
        Q = self.w_q(q).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        K = self.w_k(k).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        V = self.w_v(v).view(batch_size, -1, self.num_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_head)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)
        attn = self.dropout(F.softmax(scores, dim=-1))
        context = torch.matmul(attn, V)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(context)


def clones(module, N):
    return nn.ModuleList([copy.deepcopy(module) for _ in range(N)])


class EncoderLayer(nn.Module):
    def __init__(self, size, self_attn, feed_forward, dropout):
        super().__init__()
        self.self_attn = self_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 2)
        self.size = size
    def forward(self, x, mask):
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, mask))
        return self.sublayer[1](x, self.feed_forward)


class Encoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    def forward(self, x, mask):
        for layer in self.layers:
            x = layer(x, mask)
        return self.norm(x)

### 实现：Decoder 查询 memory，Generator 输出词表分数

`DecoderLayer` 中，第一段注意力接收 `x, x, x`，第二段接收 `x, memory, memory`：这两行就把自注意力和交叉注意力的分工写出来了。

`Generator` 用 `Linear(D,V)` 将隐藏宽度 D 映射到词表大小 V，再做 `log_softmax`。这份输出可以直接交给 `nll_loss`；它会在下一篇训练中派上用场。

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, size, self_attn, src_attn, feed_forward, dropout):
        super().__init__()
        self.size = size
        self.self_attn = self_attn
        self.src_attn = src_attn
        self.feed_forward = feed_forward
        self.sublayer = clones(SublayerConnection(size, dropout), 3)
    def forward(self, x, memory, src_mask, tgt_mask):
        m = memory
        x = self.sublayer[0](x, lambda x: self.self_attn(x, x, x, tgt_mask))
        x = self.sublayer[1](x, lambda x: self.src_attn(x, m, m, src_mask))
        return self.sublayer[2](x, self.feed_forward)


class Decoder(nn.Module):
    def __init__(self, layer, N):
        super().__init__()
        self.layers = clones(layer, N)
        self.norm = LayerNorm(layer.size)
    def forward(self, x, memory, src_mask, tgt_mask):
        for layer in self.layers:
            x = layer(x, memory, src_mask, tgt_mask)
        return self.norm(x)


class Generator(nn.Module):
    def __init__(self, d_model, vocab):
        super().__init__()
        self.proj = nn.Linear(d_model, vocab)
    def forward(self, x):
        return F.log_softmax(self.proj(x), dim=-1)


class Transformer(nn.Module):
    def __init__(self, encoder, decoder, src_embed, tgt_embed, generator):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_embed = src_embed
        self.tgt_embed = tgt_embed
        self.generator = generator

    def forward(self, src, tgt, src_mask, tgt_mask):
        memory = self.encode(src, src_mask)
        output = self.decode(memory, src_mask, tgt, tgt_mask)
        return output

    def encode(self, src, src_mask):
        return self.encoder(self.src_embed(src), src_mask)

    def decode(self, memory, src_mask, tgt, tgt_mask):
        return self.decoder(self.tgt_embed(tgt), memory, src_mask, tgt_mask)


def make_model(src_vocab, tgt_vocab, N=6, d_model=512, d_ff=2048, h=8, dropout=0.1):
    c = copy.deepcopy
    attn = MultiHeadAttention(d_model, h)
    ff = PositionWiseFeedForward(d_model, d_ff, dropout)
    position = PositionalEncoding(d_model, dropout)
    model = Transformer(
        Encoder(EncoderLayer(d_model, c(attn), c(ff), dropout), N),
        Decoder(DecoderLayer(d_model, c(attn), c(attn), c(ff), dropout), N),
        nn.Sequential(Embeddings(d_model, src_vocab), c(position)),
        nn.Sequential(Embeddings(d_model, tgt_vocab), c(position)),
        Generator(d_model, tgt_vocab)
    )
    for p in model.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)
    return model

In [ ]:
model = make_model(16, 16, N=1, d_model=8, d_ff=32, h=2, dropout=0.0).eval()
src = torch.tensor([[1, 2, 3]])
tgt = torch.tensor([[0, 3, 2]])
src_mask = torch.ones(1, 1, 1, src.size(1), dtype=torch.bool)
tgt_mask = torch.ones(tgt.size(1), tgt.size(1), dtype=torch.bool).tril()[None, None]
with torch.no_grad():
    memory = model.encode(src, src_mask)
    hidden = model.decode(memory, src_mask, tgt, tgt_mask)
    log_probs = model.generator(hidden)
print("Source tokens:", src.shape)
print("Encoder memory:", memory.shape)
print("Decoder hidden:", hidden.shape)
print("Vocabulary log probabilities:", log_probs.shape)
print("Probability sum:", log_probs.exp().sum(dim=-1))

本例中，隐藏状态的最后一维是 8，词表输出的最后一维是 16。前者是模型加工信息的宽度，后者对应 16 个候选 token。

把对数概率 `exp()` 回来再求和，每个位置得到约 1：现在模型已经为“下一个 token”准备好一份完整的概率分布。

## 用一趟数据流复习

<details>
<summary>FFN 和注意力分别忙什么？</summary>

注意力跨位置收集内容，FFN 在每个位置加工特征。前者像开会，后者像回到工位消化信息。

</details>

<details>
<summary>LayerNorm 的统计量来自哪里？</summary>

来自当前 token 的一行特征。不同句子、不同位置各算各的均值和方差。

</details>

<details>
<summary>Decoder 从哪里拿到原文信息？</summary>

交叉注意力使用目标侧状态作为 Q，Encoder memory 作为 K/V，按匹配权重汇总原文内容。

</details>

模型已经搭好，下一篇就让它开始学：[逆序任务与自回归生成](03_逆序任务与自回归生成.ipynb)。